# CSIS3754 Exam Q2 – 2025
## Soccer Match Man of the Match Classifier

## 2.1 — Import dataset

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

results = pd.read_csv('game_results.csv')
print(results.head())

## 2.2 — Brief summary

In [ ]:
print('Records, Features:', results.shape)
print(results.dtypes)

## 2.3 — Probability of 1st goal: first half vs second half

In [ ]:
total = len(results)
first_half  = results[results['First Goal'] <= 45].shape[0]
second_half = results[results['First Goal'] >  45].shape[0]

p_first  = round(first_half  / total * 100, 2)
p_second = round(second_half / total * 100, 2)

print(f'First half probability:  {p_first}%')
print(f'Second half probability: {p_second}%')

plt.figure(figsize=(6, 6))
plt.pie([p_first, p_second],
        labels=['First Half', 'Second Half'],
        autopct=lambda p: f'{p:.2f}%',
        startangle=90)
plt.title('1st Goal: First Half vs Second Half')
plt.show()

## 2.4 — Handle missing values

In [ ]:
# Check missing values
missing = results.isnull().sum()
missing_pct = (missing / len(results) * 100).round(2)
print(missing_pct[missing_pct > 0])

In [ ]:
# Handle missing values
for col in results.columns:
    if results[col].isnull().sum() > 0:
        if results[col].dtype == 'object':
            results[col].fillna(results[col].mode()[0], inplace=True)
            print(f"Filled '{col}' with mode (categorical column)")
        else:
            results[col].fillna(results[col].median(), inplace=True)
            print(f"Filled '{col}' with median (numeric column)")

print('\nRemaining missing values:')
print(results.isnull().sum().sum())

## 2.5 — Remove Date feature

In [ ]:
results = results.drop(columns=['Date'])
print(results.columns.tolist())

## 2.6 — Convert text to numeric

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encode target: Man of the Match
le = LabelEncoder()
results['Man of the Match'] = le.fit_transform(results['Man of the Match'])
print('Man of the Match classes:', le.classes_)

# Encode input features
text_cols = results.select_dtypes(include=['object']).columns.tolist()

for col in text_cols:
    unique_vals = results[col].nunique()
    if unique_vals < 10:
        le_col = LabelEncoder()
        results[col] = le_col.fit_transform(results[col])
        print(f"Label encoded (categorical): '{col}' ({unique_vals} unique)")
    else:
        dummies = pd.get_dummies(results[col], prefix=col)
        results = pd.concat([results.drop(columns=[col]), dummies], axis=1)
        print(f"One-hot encoded: '{col}' ({unique_vals} unique)")

results = results.astype({col: int for col in results.select_dtypes('bool').columns})
print('\nRemaining object columns:', results.select_dtypes(include=['object']).columns.tolist())

## 2.7 — Correlation with Man of the Match

In [ ]:
# Use only non-one-hot columns for correlation
ohe_cols = [c for c in results.columns if '_' in c and results[c].nunique() == 2]
core_cols = [c for c in results.columns if c not in ohe_cols]
corr_df = results[core_cols]

corr_vector = corr_df.corr()['Man of the Match'].sort_values(ascending=False)
print(corr_vector)

plt.figure(figsize=(10, 8))
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
print("""
2.7.1 Inference from correlation heatmap:

Features with higher positive correlation values indicate a stronger
linear relationship with Man of the Match. These features are the most
influential in determining which team's player receives the award.
For example, if goal difference or shots on target show high correlation,
it suggests winning or dominant teams are more likely to produce the
man of the match. Features close to 0 have little predictive value.
""")

## 2.8 — X, y split and train/test split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

results = results.select_dtypes(exclude=['datetime64[ns]'])

X = results.drop(columns=['Man of the Match'])
y = results['Man of the Match']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('X_train:', X_train.shape)
print('y_train:', y_train.shape)
print('X_test: ', X_test.shape)
print('y_test: ', y_test.shape)

## 2.9 — Train classifiers with k=10 fold CV + learning curves

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, cross_val_score, learning_curve

models = [
    ('NB',  GaussianNB()),
    ('DT',  DecisionTreeClassifier()),
    ('LR',  LogisticRegression(solver='lbfgs', max_iter=1000))
]

kfold = KFold(n_splits=10, shuffle=True, random_state=42)
results_acc = []
results_f1  = []
names       = []

def plot_learning_curve(model, name, X, y, cv):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y, cv=cv, scoring='accuracy',
        train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
    )
    train_mean = train_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    val_mean   = val_scores.mean(axis=1)
    val_std    = val_scores.std(axis=1)
    plt.figure(figsize=(8, 5))
    plt.plot(train_sizes, train_mean, label='Training score', color='blue')
    plt.fill_between(train_sizes, train_mean-train_std, train_mean+train_std, alpha=0.1, color='blue')
    plt.plot(train_sizes, val_mean, label='CV score', color='orange')
    plt.fill_between(train_sizes, val_mean-val_std, val_mean+val_std, alpha=0.1, color='orange')
    plt.title(f'Learning Curve — {name}')
    plt.xlabel('Training Size')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.tight_layout()
    plt.show()

for name, model in models:
    acc_scores = cross_val_score(model, X_train_scaled, y_train, cv=kfold, scoring='accuracy')
    f1_scores  = cross_val_score(model, X_train_scaled, y_train, cv=kfold, scoring='f1_weighted')
    results_acc.append(acc_scores)
    results_f1.append(f1_scores)
    names.append(name)
    print(f'{name} (acc): {acc_scores.mean():.4f}')
    print(f'{name} (f1):  {f1_scores.mean():.4f}')
    print(f'{name} done\n')
    plot_learning_curve(model, name, X_train_scaled, y_train, kfold)

## 2.10 — Best model predictions

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

f1_means   = [s.mean() for s in results_f1]
best_index = f1_means.index(max(f1_means))
best_name  = names[best_index]
best_clf   = dict(models)[best_name]
print(f'Best model: {best_name} (F1 = {f1_means[best_index]:.4f})')

best_clf.fit(X_train_scaled, y_train)
y_pred = best_clf.predict(X_test_scaled)

print(f'\nTest Accuracy: {accuracy_score(y_test, y_pred):.4f}\n')

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix — {best_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

print('Classification Report:')
print(classification_report(y_test, y_pred))

## 2.11 — Dummy classifier comparison

In [ ]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train_scaled, y_train)
y_dummy = dummy.predict(X_test_scaled)

dummy_acc = accuracy_score(y_test, y_dummy)
best_acc  = accuracy_score(y_test, y_pred)

print(f'Dummy classifier accuracy: {dummy_acc:.4f}')
print(f'Best model accuracy:       {best_acc:.4f}')
print(f'Improvement:               {best_acc - dummy_acc:.4f}')

print(f"""
Comparison Discussion:
The dummy classifier predicts the most frequent class every time,
achieving an accuracy of {dummy_acc:.4f}. The best model ({best_name})
achieves {best_acc:.4f}. An improvement of {best_acc - dummy_acc:.4f}
indicates {'a meaningful improvement' if best_acc - dummy_acc > 0.05 else 'a marginal improvement'}
over the baseline. {'The model has learned useful patterns from the features.' if best_acc - dummy_acc > 0.05 else 'The features may not provide strong enough signal to significantly outperform random guessing.'}
""")